In [1]:
import pymysql
import pandas as pd
import json
import os
import sys
import requests
sys.path.append("/home/trade_core")

In [3]:
with open('/home/trade_core/trade_core_config.json') as config_file:
    config = json.load(config_file)

node_db_name = config['node_settings'][config['node']]['db_settings']
node_db_setting_dict = config['database_setting'][node_db_name]

In [4]:
import os
import sys
import psycopg2
# upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# sys.path.append(upper_dir)
# from loggers.logger import KimpBotLogger

class InitDBClient:
    def __init__(self, host, port, user, passwd, database, create_database=True, logging_dir=None):
        self.logger = None
        self.host = host
        self.port = port
        self.user = user
        self.passwd = passwd
        self.database = database
        if logging_dir is not None:
            self.logger = KimpBotLogger("InitDBClient", logging_dir).logger
        if create_database:
            conn = psycopg2.connect(host=host, port=port, user=user, password=passwd, database='postgres')
            conn.autocommit = True
            curr = conn.cursor()
            res = curr.execute(f"SELECT 1 FROM pg_database WHERE datname='{database}'")
            res = curr.fetchone()
            if res is None:
                if self.logger is not None:
                    self.logger.info(f"InitDBClient|SCHEMA: {database} does not exist. Creating...")
                else:
                    print(f"InitDBClient|SCHEMA: {database} does not exist. Creating...")
                curr.execute(f"CREATE DATABASE {database}")
            else:
                if self.logger is not None:
                    self.logger.info(f"InitDBClient|SCHEMA: {database} already exists.")
                else:
                    print(f"InitDBClient|SCHEMA: {database} already exists.")
            curr.close()
            conn.close()

    def get_conn(self):
        conn = psycopg2.connect(host=self.host, port=self.port, user=self.user, password=self.passwd, database=self.database)
        return conn

    def create_all_table(self):
        self.create_user_info()
        self.create_exchange_config()

    def check_table_exist(self, table_name):
        query = f"""
        SELECT EXISTS (
           SELECT FROM information_schema.tables 
           WHERE  table_schema = 'public'
           AND table_name   = '{table_name}'
        );
        """
        conn = self.get_conn()
        curr = conn.cursor()
        curr.execute(query)
        exist_flag = curr.fetchone()[0]
        return exist_flag
    
    def create_user_info(self, table_name='user_info'):
        # First check whether the table exists
        if self.check_table_exist(table_name):
            if self.logger is not None:
                self.logger.info(f"InitDBClient|TABLE: {table_name} already exists.")
            else:
                print(f"InitDBClient|TABLE: {table_name} already exists.")
            return
        query = f"""
            CREATE TABLE IF NOT EXISTS {table_name}
            (
                id SERIAL PRIMARY KEY,
                user_uuid TEXT NOT NULL UNIQUE,
                email TEXT,
                telegram_id BIGINT,
                telegram_name TEXT,
                registered_datetime TIMESTAMP,
                status TEXT,
                alarm_num INTEGER,
                alarm_period INTEGER,
                remark TEXT
            )"""
        conn = self.get_conn()
        curr = conn.cursor()
        curr.execute(query)
        conn.commit()
        curr.close()
        conn.close()
        if self.logger is not None:
            self.logger.info(f"InitDBClient|TABLE: {table_name} created.")
        else:
            print(f"InitDBClient|TABLE: {table_name} created.")

    def create_exchange_config(self, table_name='exchange_config'):
        # First check whether the table exists
        if self.check_table_exist(table_name):
            if self.logger is not None:
                self.logger.info(f"InitDBClient|TABLE: {table_name} already exists.")
            else:
                print(f"InitDBClient|TABLE: {table_name} already exists.")
            return
        query = f"""
            CREATE TABLE IF NOT EXISTS {table_name}
            (
                user_uuid TEXT NOT NULL, 
                registered_datetime TIMESTAMP,
                service_datetime_end TIMESTAMP, -- PostgreSQL uses TIMESTAMP for both date and time
                target_market_code TEXT,
                origin_market_code TEXT,
                target_market_uid TEXT,
                origin_market_uid TEXT,
                target_market_referral_use SMALLINT,
                origin_market_referral_use SMALLINT,
                target_market_cross SMALLINT,
                target_market_leverage INTEGER,
                origin_market_cross SMALLINT,
                origin_market_leverage INTEGER,
                target_market_margin_call SMALLINT,
                origin_market_margin_call SMALLINT,
                target_market_safe_reverse SMALLINT,
                origin_market_safe_reverse SMALLINT,
                target_market_risk_threshold_p REAL,
                origin_market_risk_threshold_p REAL,
                repeat_limit_p REAL,
                repeat_limit_direction TEXT,
                repeat_num_limit INTEGER,
                on_off SMALLINT,
                remark TEXT,
                PRIMARY KEY (user_uuid),
                FOREIGN KEY (user_uuid) REFERENCES user_info(user_uuid)
                    ON DELETE CASCADE
                    ON UPDATE CASCADE
            )"""
        conn = self.get_conn()
        curr = conn.cursor()
        curr.execute(query)
        conn.commit()
        curr.close()
        conn.close()
        if self.logger is not None:
            self.logger.info(f"InitDBClient|TABLE: {table_name} created.")
        else:
            print(f"InitDBClient|TABLE: {table_name} created.")

    def create_trade(self, table_name='trade'):
        # First check whether the table exists
        if self.check_table_exist(table_name):
            if self.logger is not None:
                self.logger.info(f"InitDBClient|TABLE: {table_name} already exists.")
            else:
                print(f"InitDBClient|TABLE: {table_name} already exists.")
            return
        query = f"""
            CREATE TABLE IF NOT EXISTS {table_name}
            (
                id SERIAL PRIMARY KEY,
                user_uuid TEXT NOT NULL,
                last_update_datetime TIMESTAMP,
                uuid TEXT NOT NULL,
                connected_repeat_uuid TEXT,
                base_asset TEXT NOT NULL,
                usdt_conversion BOOLEAN NOT NULL,
                target_market_code TEXT NOT NULL,
                origin_market_code TEXT NOT NULL,
                low REAL NOT NULL,
                high REAL NOT NULL,
                trigger_switch SMALLINT,
                trade_switch SMALLINT,
                trade_capital REAL,
                enter_target_market_order_id TEXT,
                enter_origin_market_order_id TEXT,
                exit_target_market_order_id TEXT,
                exit_origin_market_order_id TEXT,
                status TEXT NOT NULL,
                remark TEXT,
                FOREIGN KEY (user_uuid) REFERENCES user_info(user_uuid)
                    ON DELETE CASCADE
                    ON UPDATE CASCADE
            )"""
        conn = self.get_conn()
        curr = conn.cursor()
        curr.execute(query)
        conn.commit()
        curr.close()
        conn.close()
        if self.logger is not None:
            self.logger.info(f"InitDBClient|TABLE: {table_name} created.")
        else:
            print(f"InitDBClient|TABLE: {table_name} created.")

    def create_repeat_trade(self, table_name='repeat_trade'):
        # First check whether the table exists
        if self.check_table_exist(table_name):
            if self.logger is not None:
                self.logger.info(f"InitDBClient|TABLE: {table_name} already exists.")
            else:
                print(f"InitDBClient|TABLE: {table_name} already exists.")
            return
        query = f"""
            CREATE TABLE IF NOT EXISTS {table_name}
            (
                id SERIAL PRIMARY KEY,
                user_uuid TEXT NOT NULL,
                last_update_datetime TIMESTAMP,
                uuid TEXT NOT NULL,
                base_asset TEXT NOT NULL,
                usdt_conversion BOOLEAN NOT NULL,
                auto_low REAL,
                auto_high REAL,
                pauto_num REAL,
                switch SMALLINT,
                auto_trade_switch SMALLINT,
                auto_trade_capital REAL,
                enter_target_market_order_id TEXT,
                enter_origin_market_order_id TEXT,
                exit_target_market_order_id TEXT,
                exit_origin_market_order_id TEXT,
                status TEXT NOT NULL,
                remark TEXT,
                FOREIGN KEY (user_uuid) REFERENCES user_info(user_uuid)
                    ON DELETE CASCADE
                    ON UPDATE CASCADE
            )"""
        conn = self.get_conn()
        curr = conn.cursor()
        curr.execute(query)
        conn.commit()
        curr.close()
        conn.close()
        if self.logger is not None:
            self.logger.info(f"InitDBClient|TABLE: {table_name} created.")
        else:
            print(f"InitDBClient|TABLE: {table_name} created.")

In [5]:
db_client = InitDBClient(**{**node_db_setting_dict, 'database': 'trade_core'})

InitDBClient|SCHEMA: trade_core already exists.


In [6]:
db_client.create_all_table()

InitDBClient|TABLE: user_info created.
InitDBClient|TABLE: exchange_config created.


In [7]:
response = requests.get("https://arbicrypto.net/api/users/users/")

In [9]:
response.json(

{'results': [{'uuid': 'fdda1fe4-cff4-4678-9037-0217b2d4b11a',
   'email': 'lhouernestine@gmail.com',
   'username': '엔타인',
   'first_name': 'Lhou Ernestine',
   'last_name': 'Lariosa',
   'role': 'user',
   'is_active': True,
   'date_joined': '2023-12-11T07:22:09+0000',
   'profile': {'referral': None,
    'level': 1,
    'points': 0,
    'picture': 'https://lh3.googleusercontent.com/a/ACg8ocLCftzM7HUn1cl0zb3JOVgWKJHgmBcSmEWKYRpIEKw1oBo=s96-c',
    'user': 'fdda1fe4-cff4-4678-9037-0217b2d4b11a'},
   'favorite_assets': [],
   'arbot_config': None},
  {'uuid': '5273f669-06d2-47cd-8721-3fee20665e8b',
   'email': 'superuser@halo-soft.net',
   'username': 'superuser',
   'first_name': 'Super',
   'last_name': 'User',
   'role': 'internal',
   'is_active': True,
   'date_joined': '2023-12-11T07:49:56+0000',
   'profile': {'referral': None,
    'level': 1,
    'points': 0,
    'picture': None,
    'user': '5273f669-06d2-47cd-8721-3fee20665e8b'},
   'favorite_assets': [],
   'arbot_config': N